In [0]:

import pandas as pd
from pyspark.sql import functions as F

# Step 1: Aggregate using Spark first
daily_spark = (
    spark.table("workspace.gold.agg_daily_pickup_zone")
    .groupBy("pickup_date_key")
    .agg(
        F.sum("accepted_trip_count").alias("total_trips"),
        F.sum("total_recorded_charges").alias("total_charges")
    )
    .orderBy("pickup_date_key")
)

# Confirm our result is small before collecting
assert daily_spark.count() == 31

# Step 2: Convert small Spark result into Pandas
daily_pd = daily_spark.toPandas()

# Step 3: Convert date keys into actual dates
daily_pd["date"] = pd.to_datetime(
    daily_pd["pickup_date_key"].astype(str),
    format="%Y%m%d"
)

# Step 4: Calculate a 7-day rolling average
daily_pd["seven_day_avg_trips"] = (
    daily_pd["total_trips"]
    .rolling(window=7, min_periods=1)
    .mean()
)

# Step 5: Compare weekday and weekend demand
daily_pd["day_of_week"] = daily_pd["date"].dt.day_name()

daily_pd["is_weekend"] = (
    daily_pd["date"].dt.dayofweek >= 5
)

display(
    daily_pd[
        ["date", "total_trips",
         "seven_day_avg_trips",
         "day_of_week", "is_weekend"]
    ]
)
